# Azure ML End-to-End: Iris Dataset

## What we build

```
LOCAL NOTEBOOK                     AZURE ML (runs on remote cluster)
═══════════════                    ══════════════════════════════════

1. Setup          →  MLClient connects your notebook to AML workspace
2. Compute        →  Create a CPU cluster (spins up only when jobs run)
3. Environment    →  Register conda env (Docker image + packages)
4. Data           →  Upload iris.csv → AML Data Asset
5. Components     →  Define data_prep step + train step
6. Pipeline       →  Chain steps: data_prep → train
7. Run pipeline   →  AML executes on cluster, MLflow tracks everything
8. Endpoint       →  Deploy trained model as REST API
9. Test endpoint  →  Send a prediction request
```

## MLflow in each step

| Step | What MLflow does |
|------|------------------|
| data_prep | logs params (test_ratio) + metrics (num_rows, train_rows) |
| train | autolog logs all sklearn params + test_accuracy automatically |
| model registry | mlflow registers model artifact → AML Model Registry |
| endpoint | model is loaded from MLflow artifact path for serving |

## Folder structure
```
aml_iris_project/
├── aml_iris_end_to_end.ipynb   ← you are here
├── dependencies/
│   └── conda.yml               ← packages for the remote cluster
└── components/
    ├── data_prep/
    │   └── data_prep.py        ← step 1: split data, log to mlflow
    └── train/
        ├── train.py            ← step 2: train model, mlflow autolog
        └── train.yml           ← component definition (YAML)
```

---
## Step 0 — Install SDK (run once, then restart kernel)

In [ ]:
# %pip install azure-ai-ml azure-identity
#https://learn.microsoft.com/en-us/azure/machine-learning/tutorial-azure-ml-in-a-day?view=azureml-api-2

---
## Step 1 — Connect to AML Workspace

`MLClient` is the entry point to everything in AML. It authenticates you and gives you
handles to compute, data, environments, jobs, models, and endpoints.

**Find your values in terminal:**
```bash
az login
az account show --query id --output tsv          # subscription_id
az group list --query "[].name" --output tsv      # resource_group
az ml workspace list --output table               # workspace
```

## If you have no workspace yet, create one first:
```bash
az group create --name rg-aml-iris --location germanywestcentral
az ml workspace create --name aml-iris-ws --resource-group rg-aml-iris --location germanywestcentral
```


In [ ]:
#!az group create --name rg-aml-iris --location germanywestcentral
#!az ml workspace create --name aml-iris-ws --resource-group rg-aml-iris --location germanywestcentral

In [ ]:
#subscription_id = "ff27edb4-ed96-4cef-9121-40a489b4b897"  # az account show --query id --output tsv
#resource_group  = "aml-v2-book"                            # az group list --query "[].name" --output tsv
#workspace       = "aml2-ws"                                # az ml workspace list --output table
# ────────────────────────────────

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential

# authenticate
try:
    credential = DefaultAzureCredential()
    credential.get_token("https://management.azure.com/.default")
except Exception:
    credential = InteractiveBrowserCredential()

SUBSCRIPTION = "ff27edb4-ed96-4cef-9121-40a489b4b897"
RESOURCE_GROUP = "rg-aml-iris"
WS_NAME = "aml-iris-ws"

# Get a handle to the workspace
ml_client = MLClient(
    credential=credential,
    subscription_id=SUBSCRIPTION,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WS_NAME,
)

# Verify that the handle works correctly.
# If you get an error here, modify your SUBSCRIPTION, RESOURCE_GROUP, and WS_NAME in the previous cell.
ws = ml_client.workspaces.get(WS_NAME)
print(ws.location, ":", ws.resource_group)


---
## Step 2 — Create Compute Cluster

The cluster is **where the pipeline steps actually run** — not on your Mac.
- `min_instances=0` → cluster scales to zero when idle (no cost when not running)
- `max_instances=2` → scales up to 2 nodes for parallel steps
- The `try/except` means: reuse if it already exists, create if it doesn't

In [ ]:
from azure.ai.ml.entities import AmlCompute

COMPUTE_NAME = "cpu-cluster"

try:
    cluster = ml_client.compute.get(COMPUTE_NAME)
    print(f"Reusing existing cluster: {COMPUTE_NAME}")
except Exception:
    print(f"Creating cluster: {COMPUTE_NAME} ...")
    cluster = AmlCompute(
        name=COMPUTE_NAME,
        type="amlcompute",
        size="Standard_E2ds_v4",   # 2 vCPUs, 16 GB RAM
        min_instances=0,           # scale to zero = no cost when idle
        max_instances=2,
        idle_time_before_scale_down=120,
        tier="Dedicated",
    )
    cluster = ml_client.begin_create_or_update(cluster).result()

print(f"Cluster '{cluster.name}' is ready — size: {cluster.size}")

---
## Step 3 — Register Environment

The **environment** defines what Python packages are available on the cluster.
It's a Docker image + a conda.yml. AML builds it once, then caches it.

```
Docker base image (Ubuntu 22.04)
  └── conda.yml installs: scikit-learn, pandas, mlflow, azureml-mlflow
```

The `conda.yml` is in `./dependencies/conda.yml` — already created.

In [ ]:
import os
from azure.ai.ml.entities import Environment

dependencies_dir = "./dependencies"
custom_env_name = "aml-scikit-learn"

pipeline_job_env = Environment(
    name=custom_env_name,
    description="Custom environment for Iris pipeline",
    tags={"scikit-learn": "1.3"},
    conda_file=os.path.join(dependencies_dir, "conda.yml"),
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu22.04:latest",
    # NOTE: no fixed `version=`. AML environment versions are immutable, so
    # pinning a version makes any later conda.yml edit fail to re-register.
    # Omitting it lets AML auto-increment the version on each content change.
)
pipeline_job_env = ml_client.environments.create_or_update(pipeline_job_env)

print(
    f"Environment with name {pipeline_job_env.name} is registered to workspace, the environment version is {pipeline_job_env.version}"
)


---
## Step 4 — Upload Data Asset

We use the built-in Iris dataset from sklearn, save it as CSV locally,
then register it in AML as a **URI_FILE** data asset.

```
iris.csv (local)  →  upload to workspaceblobstore  →  AML Data Asset
                                                        name: iris-data
                                                        version: 1
```

**MLflow note**: The data asset URI is later passed into the pipeline job.
MLflow logs the data path as part of the run's input lineage.

In [ ]:
import os
import pandas as pd
from sklearn.datasets import load_iris
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

# 1. Create iris.csv locally
iris = load_iris(as_frame=True)
df = iris.frame
df.columns = [c.replace(" (cm)", "").replace(" ", "_") for c in df.columns]
df["species"] = df["target"].map({0: "setosa", 1: "versicolor", 2: "virginica"})
df = df.drop(columns=["target"])
os.makedirs("./data", exist_ok=True)
df.to_csv("./data/iris.csv", index=False)
print(f"Saved iris.csv with {len(df)} rows")

# 2. Register as AML Data Asset
try:
    iris_data = ml_client.data.get(name="iris-data", version="initial")
    print(f"Data asset URI: {iris_data.path}")
except Exception:
    data_asset_def = Data(
        path="./data/iris.csv",
        type=AssetTypes.URI_FILE,
        description="Iris dataset - 150 rows, 4 features, 3 species",
        name="iris-data",
        version="initial",
    )
    iris_data = ml_client.data.create_or_update(data_asset_def)
    print(f"Data asset URI: {iris_data.path}")


---
## Step 5 — Define Pipeline Components

A **component** = one step in the pipeline. It wraps a Python script with:
- declared inputs and outputs
- the command to run
- the environment to use

We define **two components**:

| Component | Defined as | Script |
|-----------|------------|--------|
| `data_prep` | inline `command()` in Python | `data_prep.py` |
| `train` | `train.yml` loaded with `load_component()` | `train.py` |

Both approaches do the same thing — just different syntax.

### MLflow in data_prep.py
```python
with mlflow.start_run():
    mlflow.log_param("test_ratio", 0.2)       # logs the param
    mlflow.log_metric("num_rows", 150)         # logs the metric
    mlflow.log_metric("train_rows", 120)
```

### MLflow in train.py
```python
mlflow.sklearn.autolog()   # logs ALL sklearn params+metrics automatically
with mlflow.start_run():
    model.fit(X_train, y_train)               # autolog captures this
    mlflow.log_metric("test_accuracy", 0.97)  # extra custom metric
    mlflow.sklearn.log_model(model, ...)      # saves model to registry
```

In [ ]:
from azure.ai.ml import command
from azure.ai.ml import Input, Output

data_prep_src_dir = "./components/data_prep"

data_prep_component = command(
    name="data_prep_iris",
    display_name="Data preparation for training",
    description="reads iris CSV input, splits the input to train and test",
    inputs={
        "data": Input(type="uri_file"),
        "test_train_ratio": Input(type="number"),
    },
    outputs=dict(
        train_data=Output(type="uri_folder", mode="rw_mount"),
        test_data=Output(type="uri_folder", mode="rw_mount"),
    ),
    # The source folder of the component
    code=data_prep_src_dir,
    command="""python data_prep.py \
            --data ${{inputs.data}} --test_train_ratio ${{inputs.test_train_ratio}} \
            --train_data ${{outputs.train_data}} --test_data ${{outputs.test_data}} \
            """,
    environment=f"{pipeline_job_env.name}:{pipeline_job_env.version}",
)

# Now register the component to the workspace
data_prep_component = ml_client.create_or_update(data_prep_component.component)

print(
    f"Component {data_prep_component.name} with Version {data_prep_component.version} is registered"
)


---
## Step 6 — Build the Pipeline

`@dsl.pipeline` turns a Python function into an AML pipeline.
The function body describes **how components are chained**.

```
pipeline inputs
    │
    ▼
data_prep_component(data=iris_csv, test_ratio=0.2)
    │            │
  train_data  test_data        ← outputs become inputs of next step
    │            │
    ▼            ▼
train_component(train_data=..., test_data=..., learning_rate=0.1)
    │
  model  ← registered in AML Model Registry via MLflow
```

In [ ]:
import os
from azure.ai.ml import load_component

train_src_dir = "./components/train"

# Loading the component from the yml file
train_component = load_component(source=os.path.join(train_src_dir, "train.yml"))

# Now register the component to the workspace
train_component = ml_client.create_or_update(train_component)

print(
    f"Component {train_component.name} with Version {train_component.version} is registered"
)


In [1]:
import yaml

# Open the file using a context manager
train_src_dir = "./components/train"
tainyaml = os.path.join(train_src_dir, "train.yml")
with open(tainyaml, 'r') as file:
    # Load the YAML content safely
    data = yaml.safe_load(file)


In [4]:
# Access the data like a standard Python dictionary
print(data['inputs']['train_data'])  # Outputs: localhost
print(data['outputs'])  # Outputs: 5432
print(data)

{'type': 'uri_folder'}
{'model': {'type': 'uri_folder'}}
{'name': 'train_iris_model', 'display_name': 'Train Iris Model', 'type': 'command', 'inputs': {'train_data': {'type': 'uri_folder'}, 'test_data': {'type': 'uri_folder'}, 'learning_rate': {'type': 'number', 'default': 0.1}, 'registered_model_name': {'type': 'string'}}, 'outputs': {'model': {'type': 'uri_folder'}}, 'code': '.', 'environment': 'azureml:aml-scikit-learn@latest', 'command': 'python train.py --train_data ${{inputs.train_data}} --test_data ${{inputs.test_data}} --learning_rate ${{inputs.learning_rate}} --registered_model_name ${{inputs.registered_model_name}} --model ${{outputs.model}}'}


---
## Step 7 — Submit the Pipeline Job

Here we **instantiate** the pipeline with real values, then submit it.

What happens after you submit:
1. AML uploads your `./components/` code to blob storage
2. The cluster wakes up (if at 0 nodes)
3. AML runs `data_prep.py` → waits → runs `train.py`
4. MLflow tracks everything in the background
5. Model is registered in AML Model Registry

**Track progress in AML Studio** — a link is printed after submission.

In [ ]:
# the dsl decorator tells the sdk that we are defining an Azure Machine Learning pipeline
from azure.ai.ml import dsl, Input, Output


@dsl.pipeline(
    compute=COMPUTE_NAME,
    description="E2E data_prep-train pipeline",
)
def iris_pipeline(
    pipeline_job_data_input,
    pipeline_job_test_train_ratio,
    pipeline_job_learning_rate,
    pipeline_job_registered_model_name,
):
    # using data_prep_component like a python call with its own inputs
    data_prep_job = data_prep_component(
        data=pipeline_job_data_input,
        test_train_ratio=pipeline_job_test_train_ratio,
    )

    # using train_component like a python call with its own inputs
    train_job = train_component(
        train_data=data_prep_job.outputs.train_data,  # note: using outputs from previous step
        test_data=data_prep_job.outputs.test_data,    # note: using outputs from previous step
        learning_rate=pipeline_job_learning_rate,     # note: using a pipeline input as parameter
        registered_model_name=pipeline_job_registered_model_name,
    )

    # a pipeline returns a dictionary of outputs
    # keys will code for the pipeline output identifier
    return {
        "pipeline_job_train_data": data_prep_job.outputs.train_data,
        "pipeline_job_test_data": data_prep_job.outputs.test_data,
    }


In [ ]:
registered_model_name = "iris_defaults_model"

# Let's instantiate the pipeline with the parameters of our choice
pipeline = iris_pipeline(
    pipeline_job_data_input=Input(type="uri_file", path=iris_data.path),
    pipeline_job_test_train_ratio=0.25,
    pipeline_job_learning_rate=0.05,
    pipeline_job_registered_model_name=registered_model_name,
)

# submit the pipeline job
pipeline_job = ml_client.jobs.create_or_update(
    pipeline,
    # Project's name
    experiment_name="e2e_registered_components",
)
ml_client.jobs.stream(pipeline_job.name)


---
## Step 8 — Check MLflow Metrics

After the pipeline finishes, we can retrieve the metrics that were logged
by MLflow in both steps.

**What was logged automatically:**
- `data_prep` step: `num_rows`, `num_features`, `train_rows`, `test_rows`, `test_ratio`
- `train` step: all LogisticRegression params (via autolog) + `test_accuracy`

In [ ]:
# List child jobs and their outputs
for child_job in ml_client.jobs.list(parent_job_name=pipeline_job.name):
    print(f"\n--- {child_job.display_name} ({child_job.status}) ---")
    for k, v in child_job.outputs.items():
        print(f"  output: {k} = {v}")

---
## Step 9 — Deploy Model to Online Endpoint

An **endpoint** is a REST API that serves predictions.

```
AML Model Registry
  └── iris-logistic-regression (v1)
        │
        ▼
  ManagedOnlineEndpoint  →  REST API  →  POST /score → prediction
        │
        └── ManagedOnlineDeployment (blue)
              VM: Standard_DS2_v2
              Model: loaded from MLflow artifact path
```

**MLflow note**: Because we used `mlflow.sklearn.log_model()` in train.py,
AML can deploy the model **without a custom scoring script** —
MLflow provides the `predict()` interface automatically.

In [ ]:
from azure.ai.ml.entities import (
    ManagedOnlineEndpoint,
    ManagedOnlineDeployment,
    Model,
)
from azure.ai.ml.constants import AssetTypes
import datetime

# Endpoint name must be globally unique
ENDPOINT_NAME = f"iris-endpoint-{datetime.datetime.now().strftime('%m%d%H%M')}"

# ── 1. Create the endpoint (just the URL, no model yet) ───────────────────────
endpoint = ManagedOnlineEndpoint(
    name=ENDPOINT_NAME,
    description="Iris species prediction endpoint",
    auth_mode="key",   # key-based auth — AML generates a key automatically
)
endpoint = ml_client.begin_create_or_update(endpoint).result()
print(f"Endpoint created: {endpoint.name}")
print(f"Scoring URI: {endpoint.scoring_uri}")

In [ ]:
# ── 2. Get the registered model ───────────────────────────────────────────────
# MLflow registered the model during training — we just fetch it here
model = ml_client.models.get(name=registered_model_name, label="latest")
print(f"Model: {model.name} v{model.version}")

# ── 3. Create a deployment (attaches the model to the endpoint) ───────────────
# Because we used mlflow.sklearn.log_model(), no scoring script needed!
# AML+MLflow handles the predict() interface automatically.
deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=ENDPOINT_NAME,
    model=model,
    instance_type="Standard_DS2_v2",
    instance_count=1,
)
deployment = ml_client.begin_create_or_update(deployment).result()

# Route 100% of traffic to this deployment
endpoint.traffic = {"blue": 100}
ml_client.begin_create_or_update(endpoint).result()

print(f"Deployment 'blue' is live at: {endpoint.scoring_uri}")


---
## Step 10 — Test the Endpoint

Send a real prediction request to the deployed REST API.

In [ ]:
import json, os

# Write a sample input file (MLflow model server expects this format)
sample = {
    "input_data": {
        "columns": ["sepal_length", "sepal_width", "petal_length", "petal_width"],
        "data": [
            [5.1, 3.5, 1.4, 0.2],   # should be setosa
            [6.7, 3.1, 4.7, 1.5],   # should be versicolor
            [6.3, 3.3, 6.0, 2.5],   # should be virginica
        ]
    }
}
os.makedirs("./sample", exist_ok=True)
with open("./sample/request.json", "w") as f:
    json.dump(sample, f)

# Invoke the endpoint
response = ml_client.online_endpoints.invoke(
    endpoint_name=ENDPOINT_NAME,
    deployment_name="blue",
    request_file="./sample/request.json",
)

print("Predictions:")
print(response)
# Expected: ['setosa', 'versicolor', 'virginica']

---
## Step 11 — Cleanup (optional)

Online endpoints cost money while running. Delete when done.

In [ ]:
# Delete the endpoint (also deletes all deployments under it)
ml_client.online_endpoints.begin_delete(name=ENDPOINT_NAME).result()
print(f"Endpoint {ENDPOINT_NAME} deleted.")

# Optionally delete the compute cluster
# ml_client.compute.begin_delete(COMPUTE_NAME).result()

---
## Summary: What MLflow did in each step

```
Step            MLflow action                          Where to see it
──────────────  ─────────────────────────────────────  ───────────────────────────
data_prep.py    log_param(test_ratio)                  AML Studio → Jobs → Metrics
                log_metric(num_rows, train_rows, ...)  AML Studio → Jobs → Metrics

train.py        sklearn.autolog()                      AML Studio → Jobs → Metrics
                  → logs C, max_iter, solver           (all sklearn params auto)
                  → logs training_score, etc.
                log_metric(test_accuracy)              AML Studio → Jobs → Metrics
                sklearn.log_model(registered_model_name=...)  
                  → model saved + registered           AML Studio → Models

endpoint        MLflow artifact path used for serving  AML Studio → Endpoints
                No scoring script needed because
                mlflow.sklearn.log_model() was used
```

---
## Next: operationalize this with Azure DevOps → `aml_iris_devops.ipynb`

You just ran the whole lifecycle **by hand** with the SDK — great for *learning* and *experimenting*.
In production you don't click "Run"; a **git push** does it for you, with tests and approvals.

**Notebook 2 (`aml_iris_devops.ipynb`)** rebuilds this exact pipeline as a **CI/CD pipeline in
Azure DevOps**, reusing the *same* `components/`, `conda.yml`, and `data/` — only the orchestration
changes from the **Python SDK** to **declarative CLI v2 YAML**:

```
THIS NOTEBOOK (SDK v2)              NOTEBOOK 2 (CLI v2 + Azure DevOps)
═══════════════════════            ══════════════════════════════════
ml_client.jobs.create_or_update()  →  az ml job create --file pipeline.yml
you click each cell                →  git push triggers CI → Train → Deploy
                                      with pytest gates + a prod approval
```

Open **`aml_iris_devops.ipynb`** to learn: service connections, variable groups, environments &
approvals, blue/green deployment, and the 3-stage `azure-pipelines.yml`.